### Install new libraries

In [155]:
#!pip install ddgs trafilatura

### Setup

In [156]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from pprint import pprint
from IPython.display import display, Markdown
from ddgs import DDGS
import trafilatura


# ------------------------------
# Setup
# ------------------------------

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
	raise Exception("API key is missing")
client = OpenAI()

MODEL = "gpt-4.1-mini"


### Step 1: Define the Tools

In [157]:
def search_web(query: str):
	"""Search the web using DuckDuckGo browser. Returns 3 results."""
	ddgs = DDGS()
	results = ddgs.text(query, max_results=3)
	print(f"  \u2705 Got results")
	return json.dumps(results, indent=2)

In [158]:
def fetch_url(url: str):
	"""Fetch the content of a URL using trafilatura."""
	downloaded = trafilatura.fetch_url(url)
	if downloaded:
		text =  trafilatura.extract(downloaded)
		if text:
			print(f"  \u2705 Got text: {len(text)} characters")
			return text
	print(f"  \u274C Failed to fetch or extract text.") # For debugging
	return f"Could not extract text from {url}. Try a different source." # For LLM input feedback


In [159]:
#Test search_web function with a sample query
#search_web("Apples in Madagascar") #Calling the search_web function to test it with a sample query(str)

#search_web("AI in healthcare in 2030") #Calling the search_web function to test it with a sample query(str)

In [160]:
#Test fetch_url function with a sample URL
#result = fetch_url("https://explurt.com/blog-english/madagascar-english/the-list-of-fruits-to-eat-in-madagascar/") #Calling the fetch_url function to test it with a sample URL(str)
#print(result)

### Step 2: Describe as LLM tools

In [161]:
tools = []

In [162]:
#Describe search_web function as an LLM tool
search_web_function = {
	"name": "search_web",
	"description": "Search the web using DuckDuckGo browser. Returns 3 results.",
	"parameters": {
		"type": "object",
		"properties": {
			"query": {
				"type": "string",
				"description": "The search query to find relevant websites."
			}
		},
		"required": ["query"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": search_web_function})

In [163]:
#Describe fetch_url function as an LLM tool
fetch_url_function = {
	"name": "fetch_url",
	"description": "Fetch and extract the main content from a web page.",
	"parameters": {
		"type": "object",
		"properties": {
			"url": {
				"type": "string",
				"description": "The URL of the web page to fetch and extract text from."
			}
		},
		"required": ["url"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": fetch_url_function})


In [164]:
#Check the tools list
#tools

### Step 3: Tool Call handler

In [165]:
def handle_tool_call(tool_calls):
	tool_results = []


	for tool_call in tool_calls:  # handle every tool call the model requested
		function_name = tool_call.function.name
		args = json.loads(tool_call.function.arguments)  # arguments are JSON
		print(f"  \U0001F527 Calling function {function_name} with arguments {args}") # For future debugging

		#Route to the appropriate function based on function_name
		if function_name == "search_web":
			result = search_web(args["query"])
			content = f"Search results: {result}"

		elif function_name == "fetch_url":
			result = fetch_url(args["url"])
			content = f"Fetched URL content: {result}"
		else:
			content = f"Unknown function: {function_name}"
		tool_call_result = {
			"role": "tool",
			"content": content,
			"tool_call_id": tool_call.id
		}
		tool_results.append(tool_call_result)
	return tool_results

### Step 4: The System Prompt

 This tells the LLM who it is and how to behave. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce




In [166]:

# RESEARCH_AGENT_PROMPT = """You are a research agent that can search the web and fetch content from URLs.
# You have access to the following tools:
# 1. search_web(query: str) -> str: Search the web using DuckDuckGo browser. Returns 3 results.
# 2. fetch_url(url: str) -> str: Fetch and extract the main content from a web page.
# Your task is to answer questions and provide information by using these tools when necessary.\
# If you need to search for information, use the search_web tool. If you find a relevant URL, use the fetch_url tool to extract the content.\
# Always provide accurate and concise information in your responses. If you cannot find the information, respond with 'I could not find the information you requested.\
# Do not make up information. Always cite your sources when providing information from the web.\
# If you use the fetch_url tool, provide a summary of the content you extracted."""

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: You CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
You CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""


### Step 5: The Agentic Loop

In [167]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
	"""
	Run the research agent on a topic and return the research brief.
	Args:
		topic: The topic to research
		max_iterations: Safety limit to prevent infinite loops
	Returns:
		The research brief as a string
	"""
	print(f"\n🔍 Starting research on: {topic}")
	print("=" * 60)

	# Initialize conversation messages list with system prompt + research task
	messages = [
		{"role": "system", "content": RESEARCH_AGENT_PROMPT},
		{"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n {topic}"}
	]

	# LOOP
	iteration = 0

	while iteration < max_iterations:
		iteration += 1
		print(f"\n---Iteration {iteration} ---")

		# 1. Call the LLM and get response
		response = client.chat.completions.create(
			model=MODEL,
			messages=messages,
			tools=tools,
		)

		message = response.choices[0].message
		messages.append(message)

		# 2. Check if LLM called tools
		if message.tool_calls:
			tool_results = handle_tool_call(message.tool_calls)
			messages.extend(tool_results)

		# 3. Otherwise: no tools were called, read message content
		else:
			content = message.content # No tool calls, so there's just a normal text response from the LLM
			# Check if DONE:, then return
			if content and content.startswith("DONE:"):
				research_brief = content[len("DONE:"):].strip()  # Remove "DONE:" prefix and whitespace
				print(f"  ✅ Research complete!")
				return research_brief

			# Otherwise: not yet done
			else:
				print(f"  💭 Agent is thinking...")
				pprint(content)
				# Loop continues to next iteration

		# 4. If we're entering the final iteration, force a final answer
		if iteration == max_iterations - 1:
			print("⚠️ Safety limit reached. Stopping research in next iteration.")
			messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your final research brief now, even if it is incomplete."})

	# Fallback return
	return "Research incomplete. Maximum iterations reached without finalizing brief."

### Let's Run IT!

In [168]:
brief = run_research_agent("The impact of AI on healthcare in 2030")
display(Markdown(f"### Research Brief on 'The impact of AI on healthcare in 2030'\n\n{brief}"))
#brief = run_research_agent("How to land a job in AI research as a recent graduate")
#display(Markdown(f"### Research Brief on 'How to land a job in AI research as a recent graduate'\n\n{brief}"))


🔍 Starting research on: The impact of AI on healthcare in 2030

---Iteration 1 ---
  🔧 Calling function search_web with arguments {'query': 'impact of AI on healthcare in 2030'}
  ✅ Got results

---Iteration 2 ---
  🔧 Calling function fetch_url with arguments {'url': 'https://www.irejournals.com/formatedpaper/1707774.pdf'}
  ❌ Failed to fetch or extract text.
  🔧 Calling function fetch_url with arguments {'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC13160461/'}
  ❌ Failed to fetch or extract text.
  🔧 Calling function fetch_url with arguments {'url': 'https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/'}
  ❌ Failed to fetch or extract text.

---Iteration 3 ---
  🔧 Calling function search_web with arguments {'query': 'AI impact on healthcare 2030 report'}
  ✅ Got results

---Iteration 4 ---
  🔧 Calling function fetch_url with arguments {'url': 'https://www.foxbusiness.com/technology/how-ai-could-redefine

### Research Brief on 'The impact of AI on healthcare in 2030'

Research Brief: The Impact of AI on Healthcare in 2030

Key Facts and Statistics:
- According to the IEEE Technology Megatrends 2030 Report, health technology advancements driven by AI are predicted to be among the most transformative and beneficial for humanity by 2030.
- The report highlights six major AI-driven healthcare areas: personalized medicine, genetic engineering and gene therapy, accessible early disease diagnostics and biomarkers, molecular therapeutics, protein synthesis, and understanding life.
- Within these, personalized medicine, gene therapy, and early diagnostics rank highest in impact, likelihood of success, maturity, and adoption.
- AI advances are expected to enable significant reductions in preventable chronic diseases, improve personalized clinical outcomes, enhance food safety, and expand access to high-quality nutrition.
- Key enablers underlying these advances include biotechnology, digital health, and AI-enhanced agricultural and nutritional technologies.
- Challenges include protecting privacy and security of healthcare data and overcoming cultural inertia within healthcare systems.
  
Main Themes and Arguments from Sources:
1. Transformation of Medical Practice: AI will fundamentally reshape diagnostics, treatment protocols, and patient care workflows by 2030. Precision medicine powered by AI will deliver personalized treatment plans tailored to an individual’s genetic and molecular profiles.
2. Predictive Analytics and Early Disease Detection: AI systems will analyze vast, distributed datasets forecasting health events and enabling preemptive interventions to mitigate disease progression.
3. Integration and Collaboration: AI agents will help integrate varied healthcare data and systems to promote coordinated, collaborative, and proactive care models.
4. Automation and Efficiency: Automating routine clinical tasks and decision support will reduce physician burden and optimize healthcare delivery.
5. Ethical and Regulatory Challenges: Successful AI adoption requires addressing ethical questions, ensuring data security, privacy, equity, and building trust in AI-driven healthcare.
6. AI Empowering Patients: Tools developed with AI will empower patients with personalized health insights, promoting self-care and improved health outcomes.
7. Broader Ecosystem Impacts: AI’s influence extends beyond clinical settings to include pharmaceutical innovations, nutritional guidance, and public health surveillance.

Notable Data Points:
- IEEE experts predict AI-enabled healthcare will shift from reactive treatment models to proactive health protection.
- Virtual nursing and intelligent monitoring will be vital to support aging populations in revitalized economies.
- AI-powered smart traceability systems will personalize nutrition plans at scale to curb chronic diseases.
- Robust trust, privacy, and safety standards are critical enablers emphasized by experts for successful AI deployment in healthcare.

Sources:
1. IEEE Technology Megatrends 2030 Report (as summarized in Fox Business article) - https://www.foxbusiness.com/technology/how-ai-could-redefine-healthcare-end-decade
2. "AI and Healthcare in 2030: Predictions and Pathways," Journal of Asia Pacific Medical Informatics - https://japmi.org/index.php/japmi/article/view/23

This research indicates AI’s transformative impact on healthcare by 2030 will be multi-faceted, fundamentally changing treatment, diagnostics, and patient empowerment. The pathway to these benefits hinges on ethical deployment, data security, and integration within healthcare ecosystems to realize AI’s full potential responsibly.